# Rent Price Prediction

**Tabular Regression Project** — Predict monthly rent price from property and location features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `monthly_rent`

## 2 · Project Overview

We predict **monthly rent price** for residential properties using features like square footage, bedrooms, location, building age, and amenities. Rent estimation is one of the most common real-estate regression tasks, used by tenants, landlords, and platforms like Zillow.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given a property's physical characteristics — bedrooms, bathrooms, square footage, year built, parking, pet policy, furnishing status, neighborhood, floor level, and transit access — predict the **monthly rent in USD**.

## 5 · Why This Project Matters

- **Rent estimation** is critical for tenants budgeting and landlords pricing.
- Real-estate platforms use models like this for **Zestimate-style pricing**.
- Understanding rent drivers helps **urban planners** evaluate housing affordability.
- **Location premiums** (downtown vs. suburban) quantify the value of accessibility.
- A classic regression task with strong feature engineering potential.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | 10 (bedrooms, bathrooms, sqft, year built, parking, pet, furnished, neighborhood, floor, transit score) |
| **Target** | `monthly_rent` (continuous, USD/month) |
| **Categorical** | furnished_status (3), neighborhood (5) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "monthly_rent"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 9000
bedrooms = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.05, 0.25, 0.35, 0.20, 0.10, 0.05])
bathrooms = np.clip(bedrooms + np.random.choice([-1, 0, 0, 1], n), 1, 5)
sqft = 300 + bedrooms * 250 + np.random.normal(0, 80, n)
sqft = np.maximum(sqft, 200)
year_built = np.random.randint(1950, 2025, n)
parking = np.random.choice([0, 1, 2], n, p=[0.3, 0.5, 0.2])
pet_allowed = np.random.choice([0, 1], n, p=[0.4, 0.6])
furnished = np.random.choice(["unfurnished", "semi", "fully"], n, p=[0.5, 0.3, 0.2])
neighborhood = np.random.choice(["downtown", "midtown", "suburban", "rural", "university"], n)
floor_level = np.random.randint(1, 30, n)
transit_score = np.random.uniform(10, 100, n)

neigh_factor = np.where(neighborhood == "downtown", 500,
               np.where(neighborhood == "midtown", 300,
               np.where(neighborhood == "university", 200,
               np.where(neighborhood == "suburban", 0, -150)))).astype(float)

rent = (sqft * 1.8 + bedrooms * 150 + bathrooms * 100
        + (year_built - 1950) * 3 + parking * 80
        + pet_allowed * 50 + transit_score * 4
        + neigh_factor + floor_level * 5
        + np.random.normal(0, 120, n))
rent = np.maximum(rent, 300)

df = pd.DataFrame({
    "bedrooms": bedrooms, "bathrooms": bathrooms.astype(int),
    "sqft": sqft.astype(int), "year_built": year_built,
    "parking_spots": parking, "pet_allowed": pet_allowed,
    "furnished_status": furnished, "neighborhood": neighborhood,
    "floor_level": floor_level, "transit_score": transit_score,
    "monthly_rent": rent
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["sqft", "bedrooms", "bathrooms",
                          "year_built", "transit_score", "floor_level"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["monthly_rent"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("monthly_rent"); ax.set_title(f"Rent vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("neighborhood")["monthly_rent"].mean().sort_values().plot(
    kind="barh", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Mean Rent by Neighborhood"); axes[0].set_xlabel("$/month")
df.groupby("furnished_status")["monthly_rent"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="darkorange", edgecolor="black")
axes[1].set_title("Mean Rent by Furnished Status"); axes[1].set_xlabel("$/month")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `monthly_rent`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel("$/month")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Std: ${df[TARGET].std():,.0f}, Range: ${df[TARGET].min():,.0f} - ${df[TARGET].max():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

X = df.drop(columns=[TARGET])
y = df[TARGET]

cat_cols = X.select_dtypes(include="object").columns.tolist()
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = oe.fit_transform(X[cat_cols])

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

X_train["price_per_sqft_proxy"] = X_train["bedrooms"] / (X_train["sqft"] + 1) * 1000
X_test["price_per_sqft_proxy"] = X_test["bedrooms"] / (X_test["sqft"] + 1) * 1000

X_train["building_age"] = 2026 - X_train["year_built"]
X_test["building_age"] = 2026 - X_test["year_built"]

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Square footage** is the strongest predictor — more space costs more.
- **Neighborhood** creates large tier differences: downtown commands ~$500+ premium over rural.
- **Transit score** positively affects rent — walkable locations are valued.
- **Bedrooms and bathrooms** have positive but partially redundant effects (correlated with sqft).
- **Building age** has a moderate negative effect — newer buildings command higher rents.

**Business takeaway:** Location and space drive rent. Landlords can justify premiums with transit access and modern amenities. Tenants benefit most by considering neighborhoods with high transit but lower downtown premiums.

## 26 · Limitations

1. Synthetic data — real rent depends on local market dynamics, vacancy rates, and regulations.
2. No geographic coordinates — same neighborhood label doesn't capture micro-location.
3. No amenities detail — gym, laundry, doorman significantly affect rent.
4. No market timing — rents fluctuate seasonally and with economic cycles.
5. No comparable sales data (comps) that real appraisers use.

## 27 · How to Improve This Project

1. Use real rental listing data (Zillow, Apartments.com, Craigslist).
2. Add geospatial features (lat/lon, distance to CBD, school district ratings).
3. Include photos-based quality scores (deep learning on listing images).
4. Add temporal features for market timing.
5. Model using hedonic pricing theory for interpretable coefficients.

## 28 · Production Considerations

- Deploy as a real-time API for rental listing platforms.
- Update model monthly with new market data.
- Provide prediction intervals with confidence bands.
- Monitor for regional market shifts (gentrification, new transit lines).

## 29 · Common Mistakes

1. Not accounting for the sqft-bedrooms collinearity.
2. Using year_built directly instead of building_age (harder to interpret).
3. Ignoring neighborhood × transit interactions.
4. Overfitting to specific neighborhoods in training data.
5. Not validating against actual comps in the target area.

## 30 · Mini Challenge / Exercises

1. Remove `neighborhood` — how much does RMSE increase?
2. Create a `sqft_per_bedroom` feature — does it improve R²?
3. One-hot encode `neighborhood` and compare with ordinal encoding.
4. Train only on downtown listings — does the model generalize to suburban?
5. Try log(monthly_rent) as target — does it improve residual normality?

## 31 · Final Summary / Key Takeaways

1. **Square footage** and **neighborhood** are the top rent predictors.
2. **Transit score** captures accessibility value beyond neighborhood labels.
3. **Building age** has a clear negative effect on rent.
4. **Gradient boosting** captures neighborhood × size interactions well.
5. **Feature engineering** (building age, density proxy) adds interpretable value.
6. **Real deployment** needs geospatial features and temporal market data.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))